# Modul P7 — Segmentasi Citra dengan Thresholding (Week 7)

**Pengolahan Citra dan Visi Komputer — Jurusan Teknologi Informasi**

Notebook ini mencakup seluruh isi Modul P7: ulasan praktikum (C.3 Global Threshold, C.4 Adaptive
Threshold, C.5 Otsu, C.6 Segmentasi berbasis wilayah, C.7 K-Means), tugas individu D1 (nomor 3–7),
serta tugas kelompok D2 (OCR KTP dengan Tesseract). Seluruh jawaban pertanyaan ditulis sebagai sel
markdown.

## Tujuan pembelajaran
1. Menjelaskan konsep segmentasi citra dan posisi thresholding di antara metode segmentasi lain.
2. Menerapkan lima jenis global threshold, adaptive threshold, dan Otsu dengan OpenCV **serta**
   mengimplementasikannya secara manual dengan NumPy.
3. Memilih metode dan nilai threshold berdasarkan histogram dan kondisi pencahayaan citra.
4. Mengevaluasi hasil segmentasi secara kuantitatif memakai **IoU, Dice, dan pixel accuracy**.
5. Menerapkan K-Means untuk segmentasi warna dan merangkai segmentasi dengan pasca-proses
   morfologi dan OCR.

## Catatan lingkungan
Notebook ini dirancang untuk berjalan di **Kaggle** (tanpa `google.colab`, tanpa GUI — semua
penayangan citra memakai matplotlib / shim `cv2_imshow`). Pencarian berkas memakai `_find_base()`
sehingga notebook juga berjalan lokal di direktori repo.

## Daftar isi
| Bagian | Isi |
|---|---|
| C.3 | Global threshold dengan pustaka OpenCV (`gradient.jpg`, thresh = 127) |
| D1.3 | Lima global threshold **manual** (thresh = 200) |
| C.4 | Adaptive threshold dengan pustaka (`sudoku-original.jpg`) |
| D1.4 | Adaptive threshold **manual** (blok 11, C = 2, sigma = 2) |
| C.5 | Otsu dengan pustaka (`lena_gs_lc2.jpg`, `noisy2.png`) |
| D1.5 | Otsu **manual** (`lena_gs_lc2.jpg`, `ktp.jpg`) + jawaban pertanyaan |
| — | Evaluasi kuantitatif IoU / Dice / pixel accuracy |
| C.6 + D1.6 | Region growing manual (3 seed × 3 nilai T) dan `skimage.segmentation.flood()` |
| C.7 + D1.7 | K-Means pada `lily.jpg` dan ekstraksi warna biru pada `ktp.jpg` |
| D2 | OCR KTP: berwarna, Otsu, adaptive + tabel perbandingan dan analisis |

In [ ]:
# ==== KONFIGURASI LINGKUNGAN (jalan di Kaggle maupun lokal) ====
import os, sys, glob, math
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

def _find_base():
    for p in ('/kaggle/input/pcvk-images',
              '/kaggle/input/datasets/singgihwahyupermana/pcvk-images'):
        if os.path.isdir(p):
            return p
    _hit = glob.glob('/kaggle/input/**/pcvk-images', recursive=True)
    if _hit:
        return _hit[0]
    d = os.getcwd()
    for _ in range(6):
        if all(os.path.isdir(os.path.join(d, f'week{i}')) for i in (1, 2, 3)):
            return d
        d = os.path.dirname(d)
    return None

BASE = _find_base()
WEEK = os.path.join(BASE, 'week7')   # <-- ganti angka minggu sesuai notebook

def show(img, title='', cmap=None, figsize=(6, 4)):
    plt.figure(figsize=figsize)
    if img.ndim == 3:
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(title); plt.axis('off'); plt.show()

# Shim cv2_imshow (Colab) -> Kaggle
try:
    from google.colab.patches import cv2_imshow
except Exception:
    def cv2_imshow(img):
        show(img)

print('BASE :', BASE)
print('WEEK :', WEEK)
print('files:', sorted(os.listdir(WEEK)) if WEEK and os.path.isdir(WEEK) else 'MISSING')

---
## C.3 Global Threshold dengan pustaka OpenCV

Thresholding global memakai **satu** nilai ambang $T$ untuk seluruh citra:

$$g(x,y)=\begin{cases}1 & f(x,y) > T\\ 0 & f(x,y) \le T\end{cases}$$

OpenCV menyediakan lima varian pada `cv.threshold(src, thresh, maxval, type)`:

| Jenis | Jika piksel > thresh | Jika piksel ≤ thresh |
|---|---|---|
| `THRESH_BINARY` | menjadi `maxval` | menjadi 0 |
| `THRESH_BINARY_INV` | menjadi 0 | menjadi `maxval` |
| `THRESH_TRUNC` | menjadi `thresh` | nilai asli tetap |
| `THRESH_TOZERO` | nilai asli tetap | menjadi 0 |
| `THRESH_TOZERO_INV` | menjadi 0 | nilai asli tetap |

Praktikum di bawah memakai `gradient.jpg` dengan `thresh = 127` (sesuai contoh modul).

In [ ]:
# Praktikum C.3 -- lima global threshold dengan pustaka OpenCV
filename = os.path.join(WEEK, 'gradient.jpg')
img = cv.imread(filename)
gray_lib = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

thresh = 127      # nilai threshold yang ditentukan
maxval = 255

# 1. BINARY : piksel > 127 -> putih (255), selain itu hitam (0)
ret, thresh1 = cv.threshold(gray_lib, thresh, maxval, cv.THRESH_BINARY)
# 2. BINARY_INV : piksel > thresh -> 0, selain itu maxval
ret, thresh2 = cv.threshold(gray_lib, thresh, maxval, cv.THRESH_BINARY_INV)
# 3. TRUNC : piksel > thresh -> thresh, selain itu tetap
ret, thresh3 = cv.threshold(gray_lib, thresh, maxval, cv.THRESH_TRUNC)
# 4. TOZERO : piksel > thresh -> tetap, selain itu 0
ret, thresh4 = cv.threshold(gray_lib, thresh, maxval, cv.THRESH_TOZERO)
# 5. TOZERO_INV : piksel > thresh -> 0, selain itu tetap
ret, thresh5 = cv.threshold(gray_lib, thresh, maxval, cv.THRESH_TOZERO_INV)

titles = ['Citra Asli (gradient.jpg)', 'BINARY', 'BINARY_INV', 'TRUNC', 'TOZERO', 'TOZERO_INV']
images = [gray_lib, thresh1, thresh2, thresh3, thresh4, thresh5]

plt.figure(figsize=(15, 8))
for i in range(len(images)):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i], 'gray', interpolation='nearest', vmin=0, vmax=255)
    plt.title(titles[i] + (f' (thresh={thresh})' if i else ''))
    plt.xticks([]), plt.yticks([])
plt.suptitle('Gambar 1. Lima global threshold dengan pustaka OpenCV (gradient.jpg)')
plt.tight_layout(); plt.show()

---
## D1.3 Lima Global Threshold secara MANUAL (thresh = 200)

Tugas meminta kelima jenis threshold dibuat **manual** — jadi `cv.threshold()` tidak dipanggil
sama sekali pada sel ini. Setiap aturan di tabel modul diterjemahkan langsung menjadi operasi NumPy
(`np.where`). Citra masukan `gradient.jpg` diubah lebih dulu ke grayscale.

Untuk membuktikan implementasi manual benar, hasilnya dibandingkan dengan `cv.threshold()` (hanya
sebagai pembanding/verifikasi, bukan sebagai mesin perhitungan).

In [ ]:
# --- D1.3 : lima global threshold MANUAL, thresh = 200 ---
gray = cv.cvtColor(cv.imread(os.path.join(WEEK, 'gradient.jpg')), cv.COLOR_BGR2GRAY)

def threshold_manual(src, thresh, maxval, kind):
    """Implementasi manual 5 jenis global threshold (TANPA cv.threshold).

    src    : citra grayscale uint8
    thresh : nilai ambang T
    maxval : nilai maksimal (255)
    kind   : 'BINARY' | 'BINARY_INV' | 'TRUNC' | 'TOZERO' | 'TOZERO_INV'
    """
    s = src.astype(np.int16)
    above = s > thresh                      # f(x,y) > T  ?
    if kind == 'BINARY':                    # T: maxval , F: 0
        out = np.where(above, maxval, 0)
    elif kind == 'BINARY_INV':              # T: 0 , F: maxval
        out = np.where(above, 0, maxval)
    elif kind == 'TRUNC':                   # T: thresh , F: nilai asli
        out = np.where(above, thresh, s)
    elif kind == 'TOZERO':                  # T: nilai asli , F: 0
        out = np.where(above, s, 0)
    elif kind == 'TOZERO_INV':              # T: 0 , F: nilai asli
        out = np.where(above, 0, s)
    else:
        raise ValueError('jenis threshold tidak dikenal: ' + str(kind))
    return out.astype(np.uint8)

THRESH_M = 200
MAXVAL_M = 255
kinds = ['BINARY', 'BINARY_INV', 'TRUNC', 'TOZERO', 'TOZERO_INV']
manual_res = {k: threshold_manual(gray, THRESH_M, MAXVAL_M, k) for k in kinds}

In [ ]:
# Verifikasi (pembanding): hasil manual di atas dibandingkan dengan cv.threshold.
# Sel ini HANYA memakai cv.threshold sebagai acuan pembanding, bukan sebagai mesin perhitungan.
cv_types = {'BINARY': cv.THRESH_BINARY, 'BINARY_INV': cv.THRESH_BINARY_INV,
            'TRUNC': cv.THRESH_TRUNC, 'TOZERO': cv.THRESH_TOZERO,
            'TOZERO_INV': cv.THRESH_TOZERO_INV}
print(f'{"jenis":<12}{"manual==OpenCV":>16}{"min":>6}{"max":>6}{"unik":>8}')
for k in kinds:
    _, ref = cv.threshold(gray, THRESH_M, MAXVAL_M, cv_types[k])
    same = np.array_equal(manual_res[k], ref)
    u = len(np.unique(manual_res[k]))
    print(f'{k:<12}{str(same):>16}{manual_res[k].min():>6}{manual_res[k].max():>6}{u:>8}')

### Efek tiap aturan pada histogram

Agar perbedaan kelima aturan terlihat jelas, histogram setiap hasil dibandingkan dengan histogram
citra asli. Perhatikan: BINARY/BINARY_INV hanya menyisakan **dua** tingkat keabuan, TRUNC memotong
distribusi di `thresh`, sedangkan TOZERO/TOZERO_INV hanya memindahkan sebagian nilai ke 0.

In [ ]:
# Histogram hasil kelima threshold manual
plt.figure(figsize=(15, 8))
plt.subplot(2, 3, 1)
plt.hist(gray.ravel(), 256, range=(0, 256), color='gray')
plt.title('Histogram Citra Asli'); plt.xlabel('intensitas'); plt.ylabel('jumlah piksel')
for i, k in enumerate(kinds):
    plt.subplot(2, 3, i + 2)
    plt.hist(manual_res[k].ravel(), 256, range=(0, 256), color='steelblue')
    plt.title(f'Histogram {k} (T={THRESH_M})')
    plt.xlabel('intensitas'); plt.ylabel('jumlah piksel')
plt.suptitle('Gambar 3. Histogram citra asli dan hasil lima threshold manual')
plt.tight_layout(); plt.show()

# Ringkasan jumlah piksel per kategori
print('Piksel > T :', int((gray > THRESH_M).sum()), ' | Piksel <= T :', int((gray <= THRESH_M).sum()))
for k in kinds:
    nz = int((manual_res[k] > 0).sum())
    print(f'{k:<12} piksel bukan-nol = {nz}')

---
## C.4 Adaptive Threshold dengan pustaka OpenCV

Pada global thresholding, satu nilai $T$ dipakai untuk seluruh citra — ini gagal ketika pencahayaan
tidak merata (misalnya papan sudoku yang difoto dengan satu sisi lebih gelap). **Adaptive threshold**
memberi setiap piksel ambangnya sendiri yang dihitung dari jendela $b \times b$ di sekitarnya:

$$T(x,y) = \mu_{b}(x,y) - C \qquad\text{(mean)}\qquad T(x,y)=\mu^{G}_{b}(x,y) - C \qquad\text{(Gaussian)}$$

dengan $\mu_b$ rata-rata jendela, $\mu^{G}_b$ rata-rata berbobot Gaussian, $C$ konstanta pengurang.
Praktikum mengikuti modul: `sudoku-original.jpg`, `medianBlur(5)`, `blockSize = 11`, `C = 2`.

In [ ]:
# Praktikum C.4 -- adaptive threshold dengan pustaka OpenCV (sudoku-original.jpg)
citra = cv.medianBlur(cv.imread(os.path.join(WEEK, 'sudoku-original.jpg')), 5)
gray_sud = cv.cvtColor(citra, cv.COLOR_BGR2GRAY)

thresh = 127
ret, th1 = cv.threshold(gray_sud, thresh, 255, cv.THRESH_BINARY)
th2 = cv.adaptiveThreshold(gray_sud, 255, cv.ADAPTIVE_THRESH_MEAN_C,
                           cv.THRESH_BINARY, 11, 2)
th3 = cv.adaptiveThreshold(gray_sud, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C,
                           cv.THRESH_BINARY, 11, 2)

titles = ['Citra Asli (grayscale)', f'Global Thresholding (v = {thresh})',
          'Adaptive Mean Thresholding', 'Adaptive Gaussian Thresholding']
citra2 = [gray_sud, th1, th2, th3]

plt.figure(figsize=(12, 12))
for i in range(len(citra2)):
    plt.subplot(2, 2, i + 1), plt.imshow(citra2[i], 'gray', vmin=0, vmax=255)
    plt.title(titles[i])
    plt.xticks([]), plt.yticks([])
plt.suptitle('Gambar 4. Global vs Adaptive Threshold (pustaka OpenCV, blockSize = 11, C = 2)')
plt.tight_layout(); plt.show()

print('Rata-rata intensitas citra sudoku :', round(float(gray_sud.mean()), 2))
print('Rata-rata pada kuadran kiri-atas  :', round(float(gray_sud[:280, :280].mean()), 2))
print('Rata-rata pada kuadran kanan-bawah:', round(float(gray_sud[280:, 280:].mean()), 2))

---
## D1.4 Adaptive Threshold secara MANUAL (blok 11 × 11, C = 2, sigma = 2)

Tugas meminta adaptive mean **dan** adaptive Gaussian dibuat sendiri, **tanpa**
`cv.threshold()`, `cv.adaptiveThreshold()`, maupun fungsi filter siap pakai (`cv.blur`,
`cv.GaussianBlur`, `cv.filter2D`, `scipy.ndimage`, ...). Yang dipakai hanya operasi NumPy:

1. **Adaptive mean** — rata-rata jendela dihitung memakai *integral image* (cumulative sum 2 arah),
   sehingga kompleksitasnya $O(HW)$ untuk sembarang ukuran blok.
   $$S(x,y)=\sum_{i\le x}\sum_{j\le y} I(i,j),\qquad \mu_b(x,y)=\frac{S(x_2,y_2)-S(x_1{-}1,y_2)-S(x_2,y_1{-}1)+S(x_1{-}1,y_1{-}1)}{b^2}$$
2. **Adaptive Gaussian** — kernel 1-D $g(i)=\exp\!\big(-i^2/2\sigma^2\big)$ dinormalkan lalu
   dikonvolusi **terpisah** (baris kemudian kolom) memakai geseran array NumPy, jadi tidak ada
   fungsi konvolusi siap pakai.

Setelah ambang lokal $T(x,y)$ diperoleh, tiap piksel diuji $I(x,y) > T(x,y)$ seperti langkah
algoritma pada modul. Rata-rata lokal dibulatkan ke bilangan bulat (`np.rint`) untuk meniru
kuantisasi OpenCV pada citra `uint8`. Selain adaptive, pada sel ini juga dibuat **global threshold
manual dengan ambang 127** sebagai pembanding, sesuai ketentuan tugas.

In [ ]:
# --- D1.4 : adaptive threshold MANUAL (block 11, C = 2, sigma = 2) ---

def box_mean_filter(img, block):
    """Rata-rata jendela block x block memakai integral image (TANPA fungsi filter pustaka)."""
    b = block
    r = b // 2
    p = np.pad(img.astype(np.float64), r, mode='edge')          # padding 'edge'
    ii = np.cumsum(np.cumsum(p, axis=0), axis=1)                # integral image
    ii = np.pad(ii, ((1, 0), (1, 0)))                           # baris/kolom nol di depan
    H, W = img.shape
    S = ii[b:b + H, b:b + W] - ii[0:H, b:b + W] - ii[b:b + H, 0:W] + ii[0:H, 0:W]
    return S / float(b * b)

def gaussian_kernel_1d(n, sigma):
    """Kernel Gaussian 1-D ternormalisasi (sigma dalam satuan piksel)."""
    a = np.arange(n) - n // 2
    g = np.exp(-(a.astype(np.float64) ** 2) / (2.0 * sigma ** 2))
    return g / g.sum()

def convolve_separable(img, k):
    """Konvolusi terpisah (baris lalu kolom) manual dengan geseran array NumPy."""
    n = len(k); r = n // 2
    f = img.astype(np.float64); H, W = f.shape
    p = np.pad(f, ((r, r), (r, r)), mode='edge')
    tmp = np.zeros_like(f)
    for i in range(n):                                  # arah vertikal
        tmp += k[i] * p[i:i + H, r:r + W]
    p = np.pad(tmp, ((r, r), (r, r)), mode='edge')
    out = np.zeros_like(f)
    for j in range(n):                                  # arah horizontal
        out += k[j] * p[r:r + H, j:j + W]
    return out

def adaptive_threshold_manual(gray, block=11, C=2, sigma=None):
    """Adaptive threshold manual. sigma=None -> MEAN, sigma terisi -> GAUSSIAN."""
    if sigma is None:
        local = box_mean_filter(gray, block)             # mu_b(x,y)
    else:
        local = convolve_separable(gray, gaussian_kernel_1d(block, sigma))   # mu_G(x,y)
    local = np.rint(local).astype(np.uint8)              # kuantisasi seperti OpenCV
    T = local.astype(np.int16) - C                       # T(x,y) = mu(x,y) - C
    return ((gray.astype(np.int16) > T) * 255).astype(np.uint8)

def global_threshold_manual(gray, thresh=127):
    """Global threshold manual (ambang tetap 127)."""
    return threshold_manual(gray, thresh, 255, 'BINARY')

# citra sudoku: hanya konversi grayscale (tanpa filter siap pakai)
gray_sud_raw = cv.cvtColor(cv.imread(os.path.join(WEEK, 'sudoku-original.jpg')), cv.COLOR_BGR2GRAY)

glob127 = global_threshold_manual(gray_sud_raw, 127)
ad_mean = adaptive_threshold_manual(gray_sud_raw, block=11, C=2, sigma=None)
ad_gauss = adaptive_threshold_manual(gray_sud_raw, block=11, C=2, sigma=2)

plt.figure(figsize=(16, 9))
for i, (im, ttl) in enumerate([(gray_sud_raw, 'Citra Asli (grayscale, tanpa filter)'),
                               (glob127, 'Global threshold MANUAL (T = 127)'),
                               (ad_mean, 'Adaptive MEAN MANUAL (b=11, C=2)'),
                               (ad_gauss, 'Adaptive GAUSSIAN MANUAL (b=11, C=2, sigma=2)')]):
    plt.subplot(2, 2, i + 1); plt.imshow(im, 'gray', vmin=0, vmax=255); plt.title(ttl)
    plt.xticks([]); plt.yticks([])
plt.suptitle('Gambar 5. Global dan adaptive threshold hasil implementasi manual (sudoku-original.jpg)')
plt.tight_layout(); plt.show()

print('Piksel putih  global 127      :', int((glob127 > 0).sum()))
print('Piksel putih  adaptive mean   :', int((ad_mean > 0).sum()))
print('Piksel putih  adaptive gauss  :', int((ad_gauss > 0).sum()))

In [ ]:
# Verifikasi hasil manual terhadap pustaka OpenCV (pembanding, bukan mesin perhitungan)
ref_mean = cv.adaptiveThreshold(gray_sud_raw, 255, cv.ADAPTIVE_THRESH_MEAN_C,
                                cv.THRESH_BINARY, 11, 2)
ref_gauss = cv.adaptiveThreshold(gray_sud_raw, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv.THRESH_BINARY, 11, 2)

agree_mean = float((ad_mean == ref_mean).mean()) * 100
agree_gauss = float((ad_gauss == ref_gauss).mean()) * 100
print(f'Kesesuaian piksel manual vs OpenCV  (mean)    : {agree_mean:.2f}%')
print(f'Kesesuaian piksel manual vs OpenCV  (gaussian): {agree_gauss:.2f}%')

plt.figure(figsize=(16, 9))
for i, (im, ttl) in enumerate([(ad_mean, 'Adaptive MEAN (manual)'),
                               (ref_mean, 'Adaptive MEAN (OpenCV)'),
                               (ad_gauss, 'Adaptive GAUSSIAN (manual)'),
                               (ref_gauss, 'Adaptive GAUSSIAN (OpenCV)')]):
    plt.subplot(2, 2, i + 1); plt.imshow(im, 'gray', vmin=0, vmax=255); plt.title(ttl)
    plt.xticks([]); plt.yticks([])
plt.suptitle('Gambar 6. Perbandingan hasil manual dengan cv.adaptiveThreshold')
plt.tight_layout(); plt.show()

### Pengaruh ukuran blok dan konstanta C

Untuk memahami dua parameter adaptive threshold, ukuran blok $b$ dan $C$ divariasikan pada citra
sudoku yang sama. Ini juga memperlihatkan mengapa $b$ harus sedikit lebih besar daripada lebar
goresan huruf/garis yang ingin dipertahankan.

In [ ]:
# Studi parameter: pengaruh blockSize dan C pada adaptive mean manual
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for row, (b, C) in enumerate([(11, 2), (31, 2), (31, 10)]):
    out = adaptive_threshold_manual(gray_sud_raw, block=b, C=C, sigma=None)
    axes[row, 0].imshow(gray_sud_raw, 'gray', vmin=0, vmax=255)
    axes[row, 0].set_title('Citra asli')
    axes[row, 1].imshow(out, 'gray', vmin=0, vmax=255)
    axes[row, 1].set_title(f'MEAN b={b}, C={C}')
    axes[row, 2].imshow(out, 'gray', vmin=0, vmax=255)
    axes[row, 2].set_title(f'MEAN b={b}, C={C} (zoom)'); axes[row, 2].set_xlim(60, 260); axes[row, 2].set_ylim(300, 120)
    g = adaptive_threshold_manual(gray_sud_raw, block=b, C=C, sigma=2)
    axes[row, 3].imshow(g, 'gray', vmin=0, vmax=255)
    axes[row, 3].set_title(f'GAUSS b={b}, C={C}, sigma=2')
    for a in axes[row]: a.set_xticks([]); a.set_yticks([])
plt.suptitle('Gambar 7. Pengaruh blockSize (11 vs 31) dan C (2 vs 10) pada adaptive threshold manual')
plt.tight_layout(); plt.show()

for b, C in [(11, 2), (31, 2), (31, 10)]:
    m = adaptive_threshold_manual(gray_sud_raw, block=b, C=C, sigma=None)
    g = adaptive_threshold_manual(gray_sud_raw, block=b, C=C, sigma=2)
    print(f'b={b:>3}, C={C:>3} -> piksel putih mean={int((m > 0).sum()):>7}  gauss={int((g > 0).sum()):>7}')

---
## C.5 Otsu's Threshold (dengan pustaka)

Otsu mencari nilai ambang $t$ yang **meminimalkan variansi intra-kelas** (ekivalen dengan
memaksimalkan variansi antar-kelas):

$$\sigma_w^2(t)=\omega_0(t)\sigma_0^2(t)+\omega_1(t)\sigma_1^2(t),\qquad
\sigma_b^2(t)=\omega_0(t)\omega_1(t)\big[\mu_0(t)-\mu_1(t)\big]^2$$

Contoh pertama mengikuti modul: `lena_gs_lc2.jpg` (citra *low contrast*, rentang intensitas hanya
85–109) diblur dengan Gaussian lalu di-threshold global 127 dan Otsu. Perhatikan histogramnya: garis
merah = ambang 127, garis hitam = ambang hasil Otsu.

In [ ]:
# Praktikum C.5 (1) -- Otsu dengan pustaka pada citra low contrast
img_lena = cv.imread(os.path.join(WEEK, 'lena_gs_lc2.jpg'), 0)
blur_lena = cv.GaussianBlur(img_lena, (5, 5), 0)
thresh = 127

ret, th1 = cv.threshold(blur_lena, thresh, 255, cv.THRESH_BINARY)
ret2, th2 = cv.threshold(blur_lena, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

x = "Otsu's Thresholding dgn library (v = " + str(ret2) + ')'
titles = ['Citra Asli', f'Global Thresholding (v = {thresh})', x]
citra3 = [blur_lena, th1, th2]

plt.figure(figsize=(14, 9))
plt.subplot(2, 2, 1)
plt.hist(blur_lena.ravel(), 256, range=(50, 200))
plt.vlines(ret, 0, 40000, colors='red')      # ambang global 127
plt.vlines(ret2, 0, 40000, colors='black')   # ambang hasil Otsu
plt.title('Histogram Citra Asli (merah = 127, hitam = Otsu)')
for i in range(len(citra3)):
    plt.subplot(2, 2, i + 2), plt.imshow(citra3[i], 'gray', vmin=0, vmax=255)
    plt.title(titles[i])
    plt.xticks([]), plt.yticks([])
plt.suptitle('Gambar 8. Binary threshold vs Otsu (lena_gs_lc2.jpg)')
plt.tight_layout(); plt.show()

print('Rentang intensitas lena_gs_lc2 :', int(img_lena.min()), '-', int(img_lena.max()))
print('Piksel > 127 (threshold global):', int((blur_lena > 127).sum()), 'dari', blur_lena.size)
print('Ambang Otsu dari pustaka       :', ret2)
print('Piksel putih hasil Otsu        :', int((th2 > 0).sum()))

In [ ]:
# Praktikum C.5 (2) -- pengaruh Gaussian filter sebelum Otsu (noisy2.png)
img_n = cv.imread(os.path.join(WEEK, 'noisy2.png'), 0)
ret1, th1n = cv.threshold(img_n, 127, 255, cv.THRESH_BINARY)
ret2n, th2n = cv.threshold(img_n, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
blur_n = cv.GaussianBlur(img_n, (5, 5), 0)
ret3n, th3n = cv.threshold(blur_n, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

images = [img_n, 0, th1n,
          img_n, 0, th2n,
          blur_n, 0, th3n]
titles = ['Image Noisy Asli', 'Histogram', 'Global Thresholding (v=127)',
          'Image Noisy Asli', 'Histogram', "Otsu's Thresholding",
          'Image dengan Gaussian Filter', 'Histogram', "Otsu's Thresholding"]

plt.figure(figsize=(15, 12))
for i in range(3):
    plt.subplot(3, 3, i * 3 + 1), plt.imshow(images[i * 3], 'gray', vmin=0, vmax=255)
    plt.title(titles[i * 3]), plt.xticks([]), plt.yticks([])
    plt.subplot(3, 3, i * 3 + 2), plt.hist(images[i * 3].ravel(), 256)
    plt.title(titles[i * 3 + 1]), plt.xticks([]), plt.yticks([])
    plt.subplot(3, 3, i * 3 + 3), plt.imshow(images[i * 3 + 2], 'gray', vmin=0, vmax=255)
    plt.title(titles[i * 3 + 2]), plt.xticks([]), plt.yticks([])
plt.suptitle('Gambar 9. Otsu tanpa preprocess dan dengan Gaussian filter (noisy2.png)')
plt.tight_layout(); plt.show()

print('Ambang Otsu tanpa blur :', ret2n)
print('Ambang Otsu dengan blur:', ret3n)

---
## D1.5.A Otsu Thresholding MANUAL (tanpa pustaka)

Algoritma Otsu dihitung sendiri dari histogram memakai NumPy saja:

1. Hitung histogram $h(i)$ dan peluang $p(i)=h(i)/N$.
2. Untuk setiap kandidat ambang $t$: $\omega_0=\sum_{i<t}p(i)$, $\omega_1=1-\omega_0$,
   $\mu_0=\sum_{i<t} i\,p(i)/\omega_0$, $\mu_1=\sum_{i\ge t} i\,p(i)/\omega_1$.
3. Hitung variansi antar-kelas $\sigma_b^2(t)=\omega_0\omega_1(\mu_0-\mu_1)^2$; ambang terpilih
   adalah $t^*=\arg\max_t \sigma_b^2(t)$.

Seluruh perhitungan vektor (cumulative sum) — tidak ada loop piksel dan tidak ada `cv.threshold`.
Hasilnya dicetak dan dibandingkan dengan `cv.THRESH_OTSU`.

Pra-pemrosesan juga dibuat manual: modul menyaratkan Gaussian 5×5 sebelum Otsu, dan blur itu dihitung
oleh `gaussian_blur_manual()` (konvolusi terpisah dengan kernel $\sigma = 1{,}1$) — **tanpa**
`cv.GaussianBlur`. Nilai $\sigma = 1{,}1$ dipilih karena kernel OpenCV `GaussianBlur((5,5), 0)`
memakai $\sigma = 0{,}3\,(k-1)/2+0{,}8 = 1{,}1$; perbedaan hasil manual terhadap OpenCV hanya
≤ 1 tingkat keabuan sehingga ambang Otsu yang ditemukan tetap identik (T = 95).

In [ ]:
# --- D1.5.A : Otsu MANUAL (tanpa pustaka) ---
def otsu_manual(gray):
    """Otsu manual dari histogram. Mengembalikan (t_terpilih, kurva_between_class_variance)."""
    hist = np.bincount(gray.ravel(), minlength=256).astype(np.float64)
    N = float(gray.size)
    p = hist / N
    idx = np.arange(256, dtype=np.float64)
    w0 = np.cumsum(p)                     # omega_0(t)
    w1 = 1.0 - w0                         # omega_1(t)
    m0 = np.cumsum(idx * p)               # jumlah momen kelas belakang
    mT = m0[-1]                           # mean total
    with np.errstate(divide='ignore', invalid='ignore'):
        mu0 = m0 / w0
        mu1 = (mT - m0) / w1
    between = w0 * w1 * (mu0 - mu1) ** 2  # sigma_b^2(t)
    between[~np.isfinite(between)] = -1.0
    t = int(np.argmax(between))
    return t, between

def otsu_binarize(gray):
    t, curve = otsu_manual(gray)
    out = np.zeros_like(gray)
    out[gray > t] = 255
    return out, t, curve

def gaussian_blur_manual(gray, ksize=5, sigma=1.1):
    """Gaussian blur manual memakai konvolusi terpisah (TANPA cv.GaussianBlur)."""
    return np.rint(convolve_separable(gray, gaussian_kernel_1d(ksize, sigma))).astype(np.uint8)

# Pra-pemrosesan (sesuai modul: Gaussian 5x5) dikerjakan secara manual juga
blur_lena_m = gaussian_blur_manual(img_lena, ksize=5, sigma=1.1)
otsu_bin_lena, otsu_t_lena, curve_lena = otsu_binarize(blur_lena_m)
print("Otsu's MANUAL pada lena_gs_lc2 (blur 5x5 manual) : T =", otsu_t_lena)
print('Piksel putih hasil Otsu manual                   :', int((otsu_bin_lena > 0).sum()))

plt.figure(figsize=(16, 9))
plt.subplot(2, 2, 1)
plt.hist(blur_lena_m.ravel(), 256, range=(50, 200), color='gray')
plt.vlines(otsu_t_lena, 0, 40000, colors='black')
plt.title(f'Histogram + ambang Otsu manual (T = {otsu_t_lena})')
plt.subplot(2, 2, 2)
plt.plot(curve_lena, color='crimson')
plt.vlines(otsu_t_lena, curve_lena.min(), curve_lena.max(), colors='black', linestyles='--')
plt.title('Variansi antar-kelas $\\sigma_b^2(t)$'); plt.xlabel('t'); plt.ylabel('$\\sigma_b^2(t)$')
plt.subplot(2, 2, 3)
plt.imshow(blur_lena_m, 'gray', vmin=0, vmax=255); plt.title('Blur Gaussian 5x5 manual'); plt.axis('off')
plt.subplot(2, 2, 4)
plt.imshow(otsu_bin_lena, 'gray', vmin=0, vmax=255); plt.title(f"Otsu's MANUAL (T = {otsu_t_lena})"); plt.axis('off')
plt.suptitle('Gambar 10. Otsu manual pada lena_gs_lc2.jpg')
plt.tight_layout(); plt.show()

In [ ]:
# Verifikasi (pembanding): Otsu manual vs cv.THRESH_OTSU pada MASUKAN YANG SAMA.
# cv.threshold HANYA dipakai sebagai acuan pembanding, bukan sebagai mesin perhitungan.
ret_lib_lena, otsu_lib_lena = cv.threshold(blur_lena_m, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
print("Otsu's MANUAL  pada lena_gs_lc2 : T =", otsu_t_lena)
print("Otsu's PUSTAKA pada masukan sama: T =", int(ret_lib_lena))
print('Hasil biner identik dengan pustaka :', np.array_equal(otsu_bin_lena, otsu_lib_lena))

# Pembanding tambahan: blur bawaan OpenCV + Otsu pustaka (seperti contoh modul)
ret_lib_cv, otsu_lib_cv = cv.threshold(blur_lena, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
print("Otsu's PUSTAKA (cv.GaussianBlur + pustaka) : T =", int(ret_lib_cv))
print('Piksel biner manual vs pustaka(cv blur) berbeda :',
      int((otsu_bin_lena != otsu_lib_cv).sum()), 'piksel dari', otsu_bin_lena.size)

plt.figure(figsize=(16, 5))
for i, (im, ttl) in enumerate([(otsu_bin_lena, f"Otsu MANUAL (T={otsu_t_lena})"),
                               (otsu_lib_lena, f"Otsu PUSTAKA, masukan sama (T={int(ret_lib_lena)})"),
                               (otsu_lib_cv, f"Otsu PUSTAKA + cv blur (T={int(ret_lib_cv)})")]):
    plt.subplot(1, 3, i + 1); plt.imshow(im, 'gray', vmin=0, vmax=255); plt.title(ttl); plt.axis('off')
plt.suptitle('Gambar 10b. Otsu manual vs pustaka (lena_gs_lc2.jpg)')
plt.tight_layout(); plt.show()

In [ ]:
# Otsu manual pada lena_gs_lc2.jpg: tanpa blur vs dengan blur manual
otsu_bin_nb, otsu_t_nb, _ = otsu_binarize(img_lena)
print("Otsu's manual (tanpa blur) : T =", otsu_t_nb)
print('Piksel putih tanpa blur    :', int((otsu_bin_nb > 0).sum()))
print('Piksel putih dengan blur   :', int((otsu_bin_lena > 0).sum()))

plt.figure(figsize=(14, 5))
plt.subplot(1, 3, 1); plt.imshow(img_lena, 'gray', vmin=0, vmax=255); plt.title('Citra Asli (low contrast)'); plt.axis('off')
plt.subplot(1, 3, 2); plt.imshow(otsu_bin_nb, 'gray', vmin=0, vmax=255); plt.title(f"Otsu manual tanpa blur (T={otsu_t_nb})"); plt.axis('off')
plt.subplot(1, 3, 3); plt.imshow(otsu_bin_lena, 'gray', vmin=0, vmax=255); plt.title(f"Otsu manual dgn blur manual (T={otsu_t_lena})"); plt.axis('off')
plt.suptitle('Gambar 11. Pengaruh Gaussian blur sebelum Otsu manual (lena_gs_lc2.jpg)')
plt.tight_layout(); plt.show()

---
## D1.5.B Otsu manual pada citra KTP

Sesuai tugas, Otsu manual dijalankan juga pada citra KTP (`ktp.jpg`). Di sini perbedaan antara
**global threshold biasa (127 dan 200)** dengan **Otsu** terlihat sangat jelas, karena histogram
citra KTP tidak bimodal sempurna (latar beige, banner coklat, tulisan hitam).

In [ ]:
# --- D1.5.B : Otsu manual pada ktp.jpg, dibandingkan global threshold biasa ---
ktp = cv.imread(os.path.join(WEEK, 'ktp.jpg'))
ktp_gray = cv.cvtColor(ktp, cv.COLOR_BGR2GRAY)
ktp_blur_m = gaussian_blur_manual(ktp_gray, ksize=5, sigma=1.1)   # pra-pemrosesan manual

ktp_otsu_bin, ktp_otsu_t, ktp_curve = otsu_binarize(ktp_blur_m)     # Otsu MANUAL
ktp_glob127 = threshold_manual(ktp_blur_m, 127, 255, 'BINARY')      # global manual T=127
ktp_glob200 = threshold_manual(ktp_blur_m, 200, 255, 'BINARY')      # global manual T=200

print("Otsu's MANUAL  pada KTP : T =", ktp_otsu_t)
print('Piksel putih Otsu       :', int((ktp_otsu_bin > 0).sum()))
print('Piksel putih global 127 :', int((ktp_glob127 > 0).sum()))
print('Piksel putih global 200 :', int((ktp_glob200 > 0).sum()))

plt.figure(figsize=(18, 10))
for i, (im, ttl) in enumerate([(ktp_gray, 'Citra Asli (grayscale)'),
                               (ktp_glob127, 'Global Threshold manual (T = 127)'),
                               (ktp_glob200, 'Global Threshold manual (T = 200)'),
                               (ktp_otsu_bin, f"Otsu MANUAL (T = {ktp_otsu_t})"),
                               (255 - ktp_otsu_bin, 'Otsu manual dibalik (BINARY_INV)'),
                               (np.abs(ktp_blur_m.astype(np.int16) - ktp_otsu_t).astype(np.uint8),
                                'Selisih terhadap ambang Otsu')]):
    plt.subplot(2, 3, i + 1); plt.imshow(im, 'gray', vmin=0, vmax=255); plt.title(ttl); plt.axis('off')
plt.suptitle('Gambar 12. Global threshold vs Otsu pada ktp.jpg')
plt.tight_layout(); plt.show()

plt.figure(figsize=(10, 4))
plt.hist(ktp_blur_m.ravel(), 256, range=(0, 256), color='gray')
top = plt.gca().get_ylim()[1]
plt.vlines(127, 0, top * 0.9, colors='red')
plt.vlines(200, 0, top * 0.7, colors='orange')
plt.vlines(ktp_otsu_t, 0, top, colors='black')
plt.title(f'Gambar 13. Histogram ktp.jpg (merah=127, oranye=200, hitam=Otsu {ktp_otsu_t})')
plt.xlabel('intensitas'); plt.ylabel('jumlah piksel'); plt.show()

In [ ]:
# Verifikasi (pembanding): Otsu manual vs cv.THRESH_OTSU pada MASUKAN YANG SAMA.
# cv.threshold HANYA dipakai sebagai acuan pembanding, bukan sebagai mesin perhitungan.
ret_ktp, ktp_otsu_lib = cv.threshold(ktp_blur_m, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
print("Otsu's MANUAL  pada KTP : T =", ktp_otsu_t)
print("Otsu's PUSTAKA pada masukan sama : T =", int(ret_ktp))
print('Identik dengan pustaka  :', np.array_equal(ktp_otsu_bin, ktp_otsu_lib))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1); plt.imshow(ktp_otsu_bin, 'gray', vmin=0, vmax=255)
plt.title(f"Otsu's MANUAL (T = {ktp_otsu_t})"); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(ktp_otsu_lib, 'gray', vmin=0, vmax=255)
plt.title(f"Otsu's PUSTAKA (T = {int(ret_ktp)})"); plt.axis('off')
plt.suptitle('Gambar 12b. Otsu manual vs pustaka pada ktp.jpg')
plt.tight_layout(); plt.show()

## Jawaban — Pertanyaan D1.5

**Pertanyaan:** *Mengapa hasil tampilan Binary Thresholding semua bernilai 0 atau hitam semuanya?
Jelaskan alasannya dalam bentuk histogram dan berapa angka ambang batas (Thresh) supaya gambar Lena
ini masih terlihat?*

**Jawaban.**

1. **Rentang intensitas citra jauh di bawah ambang.** Dari keluaran sel di atas, `lena_gs_lc2.jpg`
   hanya menempati intensitas **85–109** (rentang 24 level). Aturan `THRESH_BINARY` mengubah piksel
   menjadi 255 **hanya jika** $f(x,y) > T$; karena $T = 127$ lebih besar daripada **seluruh** nilai
   piksel (nilai maksimum citra = 109), tidak ada satu pun piksel yang memenuhi syarat, sehingga
   semua piksel menjadi 0 — citra hitam total. Buktinya tercetak: `Piksel > 127 = 0 dari 262144`.
2. **Penjelasan lewat histogram.** Histogram citra (Gambar 8/10/11) berupa satu gundukan sempit di
   sekitar 85–109, sedangkan garis ambang 127 (garis merah) berada **di sebelah kanan** seluruh
   distribusi. Karena garis pemisah berada di luar data, seluruh populasi masuk ke kelas
   *background*. Ini contoh klasik kegagalan global threshold pada citra *low contrast*.
3. **Ambang agar Lena masih terlihat.** Ambang harus **di dalam** rentang data, yaitu
   $85 \le T \le 105$. Nilai yang aman adalah sekitar **95** — tepat nilai yang dipilih Otsu secara
   otomatis ($T = 95$, lihat keluaran `Otsu's manual ... T = 95`), dan konsisten dengan modul yang
   menyebut ambang Otsu ≈ 92–95 untuk citra ini. Tabel pada sel bukti menunjukkan: $T=90$ → 82,9%
   piksel putih, $T=95$ → **55,9%** putih (44,1% gelap, pemisahan paling seimbang), $T=100$ → 13,4%
   putih, dan $T=105$ hanya 76 piksel putih (praktis hitam lagi). Jadi rentang aman adalah
   **85 ≤ T ≤ 100**, dengan $T \approx 95$ sebagai pilihan terbaik.
   Sebaliknya, pada `ktp.jpg` histogramnya lebar dan multimodal, sehingga global threshold 127/200
   **tidak** menghasilkan citra hitam total — inilah bedanya dengan citra low contrast.
4. **Kesimpulan praktis.** Nilai `thresh = 127` pada modul hanya cocok untuk citra berintensitas
   penuh (0–255, mis. `gradient.jpg`). Untuk citra *low contrast* harus dipilih $T$ di dalam rentang
   data, atau lebih baik memakai **Otsu** yang mencari $T$ otomatis dari histogram.

In [ ]:
# Bukti kuantitatif jawaban di atas: uji beberapa nilai ambang pada lena_gs_lc2.jpg
print('Rentang intensitas :', int(img_lena.min()), '-', int(img_lena.max()))
print(f'{"T":>5}{"piksel putih":>16}{"% putih":>10}')
for T in [85, 90, 95, 100, 105, 109, 110, 127]:
    out = threshold_manual(img_lena, T, 255, 'BINARY')
    n = int((out > 0).sum())
    print(f'{T:>5}{n:>16}{100.0 * n / out.size:>10.2f}')

plt.figure(figsize=(14, 8))
plt.subplot(2, 4, 1)
plt.hist(img_lena.ravel(), 25, [84, 110], color='gray')
plt.title('Histogram asli (85-109)'); plt.xlabel('intensitas')
for i, T in enumerate([90, 95, 100, 105, 109, 127]):
    plt.subplot(2, 4, i + 2)
    plt.imshow(threshold_manual(img_lena, T, 255, 'BINARY'), 'gray', vmin=0, vmax=255)
    plt.title(f'T = {T}'); plt.axis('off')
plt.suptitle('Gambar 14. Bukti: ambang harus di dalam rentang 85-109 agar Lena terlihat')
plt.tight_layout(); plt.show()

---
## Evaluasi kuantitatif: IoU, Dice, dan Pixel Accuracy

Tujuan pembelajaran ke-4 adalah mengevaluasi hasil segmentasi secara kuantitatif. Ketiga metrik
didefinisikan dari himpunan piksel positif (putih) hasil segmentasi $A$ dan acuan $B$:

$$\mathrm{IoU}=\frac{|A\cap B|}{|A\cup B|},\qquad
\mathrm{Dice}=\frac{2|A\cap B|}{|A|+|B|},\qquad
\mathrm{Acc}=\frac{|A\cap B|+|A^c\cap B^c|}{N}$$

Acuan di sini adalah hasil pustaka OpenCV (manual vs pustaka), sehingga angkanya mengukur seberapa
tepat implementasi manual dan sekaligus menunjukkan bahwa Otsu lebih unggul daripada global 127 pada
citra low contrast ketika dibandingkan dengan acuan Otsu pustaka.

In [ ]:
def iou(a, b):
    A = a > 0; B = b > 0
    inter = np.logical_and(A, B).sum()
    union = np.logical_or(A, B).sum()
    return float(inter) / float(union) if union else 1.0

def dice(a, b):
    A = a > 0; B = b > 0
    inter = np.logical_and(A, B).sum()
    s = A.sum() + B.sum()
    return 2.0 * float(inter) / float(s) if s else 1.0

def pixel_accuracy(a, b):
    return float((np.asarray(a) == np.asarray(b)).mean())

rows = [
    ('Adaptive MEAN (manual vs OpenCV)', ad_mean, ref_mean),
    ('Adaptive GAUSSIAN (manual vs OpenCV)', ad_gauss, ref_gauss),
    ('Otsu manual vs Otsu pustaka (lena)', otsu_bin_lena, otsu_lib_lena),
    ('Otsu manual vs Otsu pustaka (ktp)', ktp_otsu_bin, ktp_otsu_lib),
    ('Global 127 vs Otsu pustaka (lena)', threshold_manual(img_lena, 127, 255, 'BINARY'), otsu_lib_lena),
]
print(f'{"Perbandingan":<42}{"IoU":>8}{"Dice":>8}{"Akurasi":>10}')
for name, a, b in rows:
    print(f'{name:<42}{iou(a, b):>8.4f}{dice(a, b):>8.4f}{pixel_accuracy(a, b):>10.4f}')

---
## C.6 / D1.6.A Segmentasi Berbasis Wilayah — Region Growing manual

Berbeda dari thresholding yang bertanya *"apakah nilai piksel ini di atas ambang?"*, segmentasi
berbasis wilayah bertanya *"apakah piksel ini mirip dengan tetangganya?"*. Region growing dimulai
dari sebuah **seed**, lalu tumbuh ke tetangga (di sini 8-ketetanggaan) selama memenuhi kriteria
kesamaan $|I(q) - I_{seed}| < T$.

Implementasi di bawah ditulis manual memakai *breadth-first search* (`collections.deque`) — tidak
memakai `cv.floodFill` maupun fungsi segmentasi pustaka.

**Ketentuan tugas:** 3 seed (di dalam koin, di latar, tepat di tepi koin) × 3 nilai $T$ (5, 20, 60),
ditampilkan sebagai grid 3×3. Citra yang dipakai: `skimage.data.coins()` (koin terang di latar
gelap, beberapa koin saling bersentuhan).

In [ ]:
from collections import deque
from skimage import data

coins = data.coins()
print('coins :', coins.shape, coins.dtype, '| min/max =', int(coins.min()), int(coins.max()))

def region_grow(img, seed, T, conn=8):
    """Region growing manual: BFS dari seed, terima tetangga dengan |I - I_seed| < T."""
    H, W = img.shape
    mask = np.zeros((H, W), np.uint8)
    sy, sx = seed
    val = float(img[sy, sx])
    mask[sy, sx] = 1
    queue = deque([(sy, sx)])
    offs = ([(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]
            if conn == 8 else [(-1, 0), (1, 0), (0, -1), (0, 1)])
    while queue:
        y, x = queue.popleft()
        for dy, dx in offs:
            ny, nx = y + dy, x + dx
            if 0 <= ny < H and 0 <= nx < W and not mask[ny, nx] \
               and abs(float(img[ny, nx]) - val) < T:
                mask[ny, nx] = 1
                queue.append((ny, nx))
    return mask

# 3 seed: di dalam koin, di latar, dan tepat di tepi koin
seeds = [('Seed di dalam koin (50,50)', (50, 50)),
         ('Seed di latar (270,200)', (270, 200)),
         ('Seed di tepi koin (144,146)', (144, 146))]
T_values = [5, 20, 60]

print('\nJumlah piksel region:')
print(f'{"seed":<32}' + ''.join(f'{"T=" + str(t):>10}' for t in T_values))
grid = {}
for nm, sd in seeds:
    line = f'{nm:<32}'
    for T in T_values:
        m = region_grow(coins, sd, T)
        grid[(nm, T)] = m
        line += f'{int(m.sum()):>10}'
    print(line)
print('\nNilai intensitas seed:', {nm: int(coins[sd]) for nm, sd in seeds})

# Grid 3x3 (baris = seed, kolom = T) : mask berwarna merah di atas citra asli
plt.figure(figsize=(15, 15))
for r, (nm, sd) in enumerate(seeds):
    for c, T in enumerate(T_values):
        ax = plt.subplot(3, 3, r * 3 + c + 1)
        rgb = np.stack([coins] * 3, axis=-1).copy()
        m = grid[(nm, T)] > 0
        rgb[m] = [255, 60, 60]
        ax.imshow(rgb)
        ax.plot(sd[1], sd[0], 'w+', markersize=14, markeredgewidth=2)
        ax.set_title(f'{nm.split()[1]} {nm.split()[2]} T={T}\npiksel = {int(m.sum())}', fontsize=9)
        ax.axis('off')
plt.suptitle('Gambar 15. Region growing manual pada skimage.data.coins() — baris = seed, kolom = T')
plt.tight_layout(); plt.show()

### Pembahasan grid region growing

- **Seed di dalam koin (50,50).** Intensitas seed ≈ 158. Pada $T=5$ region hanya mencakup 11 piksel
  (hanya area sekitar seed yang benar-benar seragam, karena permukaan koin bertekstur). Pada $T=20$
  region tumbuh menjadi satu koin penuh (≈ 892 piksel) tanpa keluar batas koin. Pada $T=60$ kriterianya
  terlalu longgar sehingga region "bocor" keluar dari koin (≈ 15 ribu piksel) dan ikut menelan latar.
- **Seed di latar (270,200).** Intensitas latar ≈ 27 (gelap). $T=5$ hanya memberi 9 piksel karena
  latar coins mengandung noise/gradien pencahayaan. $T=20$ sudah membanjiri latar (≈ 22.900 piksel),
  dan $T=60$ menelan hampir seluruh latar (≈ 57.400 piksel) karena batas toleransi 60 lebih besar
  daripada kontras latar-ke-koin di sebagian area.
- **Seed tepat di tepi koin (144,146).** Intensitas ≈ 177 (piksel tepi yang terang). $T=5$ → 2 piksel,
  $T=20$ → 6 piksel, $T=60$ → 1.084 piksel. Inilah ilustrasi kelemahan region growing: karena nilai
  piksel di tepi berada di **puncak gradien**, hampir tidak ada tetangga yang nilainya dalam
  toleransi, sehingga region gagal tumbuh. Jika $T$ diperbesar, region justru melompat ke area lain
  yang kebetulan nilainya mirip.
- **Kesimpulan.** Hasil region growing sangat bergantung pada posisi seed **dan** nilai $T$:
  over-segmentation pada $T$ kecil (region hanya bercak), under-segmentation/leakage pada $T$ besar.
  Ini sesuai dengan tabel perbandingan metode pada modul.

### D1.6.B `skimage.segmentation.flood()`

`flood(image, seed, tolerance=...)` menghasilkan mask region dari seed: semua piksel yang **terhubung**
dengan seed dan selisih intensitasnya terhadap intensitas seed berada dalam `tolerance` akan
disertakan. Perhatikan bahwa `tolerance` di skimage bersifat $\le$ (inklusif), berbeda dengan
implementasi manual di atas yang memakai `< T`.

In [ ]:
from skimage.segmentation import flood

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(coins, cmap='gray'); axes[0].set_title('skimage.data.coins()'); axes[0].axis('off')
for ax, (nm, sd), tol in zip(axes[1:], seeds, [20, 20, 20]):
    m = flood(coins, sd, tolerance=tol)
    rgb = np.stack([coins] * 3, axis=-1).copy()
    rgb[m] = [60, 255, 60]
    ax.imshow(rgb); ax.plot(sd[1], sd[0], 'r+', markersize=12, markeredgewidth=2)
    ax.set_title(f'{nm}\ntolerance={tol} -> {int(m.sum())} piksel', fontsize=9)
    ax.axis('off')
plt.suptitle('Gambar 16. skimage.segmentation.flood() dari tiga seed (tolerance = 20)')
plt.tight_layout(); plt.show()

print(f'{"seed":<32}{"flood tol=20":>14}{"region_grow T=20":>18}')
for nm, sd in seeds:
    print(f'{nm:<32}{int(flood(coins, sd, tolerance=20).sum()):>14}{int(grid[(nm, 20)].sum()):>18}')

### Pelengkap: Watershed Segmentation pada `coins`

Sebagai pembanding metode berbasis wilayah yang lain (dan sesuai ulasan teori C.6), watershed
dijalankan dengan **marker terkontrol**: Otsu → distance transform → marker dari puncak distance
transform → `skimage.segmentation.watershed`. Tanpa marker, watershed pada citra coins akan
menghasilkan ratusan kepingan (over-segmentation) karena setiap tekstur koin menjadi cekungan.

In [ ]:
from skimage.segmentation import watershed

den = cv.medianBlur(coins, 5)
t_coins, _ = otsu_manual(den)
bw = (den > t_coins).astype(np.uint8)
dist = cv.distanceTransform(bw, cv.DIST_L2, 3)
markers = threshold_manual(np.clip(dist, 0, 255).astype(np.uint8), int(0.4 * dist.max()), 255, 'BINARY')
n_mark, markers = cv.connectedComponents(markers.astype(np.uint8))
labels = watershed(-dist, markers, mask=bw)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(coins, cmap='gray'); axes[0].set_title('Citra coins'); axes[0].axis('off')
axes[1].imshow(bw, cmap='gray'); axes[1].set_title(f'Biner Otsu (T={t_coins})'); axes[1].axis('off')
axes[2].imshow(dist, cmap='jet'); axes[2].set_title('Distance transform'); axes[2].axis('off')
axes[3].imshow(labels, cmap='nipy_spectral'); axes[3].set_title(f'Watershed: {labels.max()} region'); axes[3].axis('off')
# Pembanding: watershed TANPA marker -> over-segmentation berat
labels_nom = watershed(-dist, markers=None, mask=bw)

plt.figure(figsize=(10, 6))
plt.subplot(1, 2, 1); plt.imshow(labels, cmap='nipy_spectral')
plt.title(f'Dengan marker: {int(labels.max())} region'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(labels_nom, cmap='nipy_spectral')
plt.title(f'Tanpa marker: {int(labels_nom.max())} region'); plt.axis('off')
plt.suptitle('Gambar 17. Watershed dengan marker dari distance transform')
plt.tight_layout(); plt.show()
print('Jumlah marker/koin terdeteksi        :', n_mark - 1)
print('Jumlah region watershed dgn marker   :', int(labels.max()))
print('Jumlah region watershed tanpa marker :', int(labels_nom.max()))

---
## C.7 Segmentasi dengan K-Means (praktikum modul)

K-Means memperlakukan citra sebagai kumpulan titik di ruang fitur (di sini warna RGB tiap piksel),
lalu membagi piksel menjadi $K$ cluster dengan meminimalkan *within-cluster sum of squares*
(inertia):

$$J=\sum_{k=1}^{K}\sum_{x_i\in C_k}\lVert x_i-\mu_k\rVert^2$$

Karena setiap piksel diganti dengan warna centroid cluster-nya, K-Means juga berfungsi sebagai
kuantisasi warna. Praktikum modul memakai `lily.jpg` dengan $K=3$. Berkas `lily.jpg` pada dataset modul ini ternyata
tersimpan **sangat gelap** (rentang intensitas hanya 0–19), sehingga sebelum di-cluster kontrasnya
direntangkan dulu dengan `cv.normalize` (min–max). Tanpa peregangan itu, seluruh piksel akan jatuh
ke satu cluster hitam dan segmentasi tidak bermakna.

In [ ]:
from sklearn.cluster import KMeans

# Citra lily.jpg pada dataset modul ini tersimpan sangat gelap (rentang intensitas 0-19),
# sehingga perlu diperiksa lebih dulu; kontras direntangkan agar segmentasi bermakna.
lily_raw = cv.cvtColor(cv.imread(os.path.join(WEEK, 'lily.jpg')), cv.COLOR_BGR2RGB)
print('lily.jpg asli  :', lily_raw.shape, '| rentang', int(lily_raw.min()), '-', int(lily_raw.max()),
      '| rata-rata', round(float(lily_raw.mean()), 2))

img_lily = cv.normalize(lily_raw, None, 0, 255, cv.NORM_MINMAX)   # peregangan kontras (min-max)
print('setelah stretch:', '| rentang', int(img_lily.min()), '-', int(img_lily.max()),
      '| rata-rata', round(float(img_lily.mean()), 2))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1); plt.imshow(lily_raw); plt.title('lily.jpg asli (sangat gelap, 0-19)'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(img_lily); plt.title('lily.jpg setelah peregangan kontras'); plt.axis('off')
plt.suptitle('Gambar 18a. Pemeriksaan awal citra lily.jpg')
plt.tight_layout(); plt.show()

Z = np.float32(img_lily.reshape((-1, 3)))
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans.fit(Z)
labels_lily = kmeans.labels_
centers = np.uint8(kmeans.cluster_centers_)
segmented_img = centers[labels_lily.flatten()].reshape(img_lily.shape)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1); plt.imshow(img_lily); plt.title('Citra (setelah stretch)'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(segmented_img); plt.title(f'Segmentasi K-Means (k={k})'); plt.axis('off')
plt.suptitle('Gambar 18. Segmentasi K-Means pada lily.jpg')
plt.tight_layout(); plt.show()

print('Centroid (RGB) :')
for i, c in enumerate(centers):
    print(f'  cluster {i}: RGB{tuple(int(v) for v in c)} -> {int((labels_lily == i).sum())} piksel')

# Metode elbow untuk memilih K
inertia = []
Ks = list(range(2, 8))
for kk in Ks:
    inertia.append(KMeans(n_clusters=kk, random_state=42, n_init=10).fit(Z).inertia_)
plt.figure(figsize=(7, 4))
plt.plot(Ks, inertia, 'o-')
plt.title('Gambar 19. Metode elbow pada lily.jpg'); plt.xlabel('K'); plt.ylabel('inertia (WCSS)')
plt.grid(True); plt.show()
print('Inertia per K:', {kk: int(v) for kk, v in zip(Ks, inertia)})

---
## D1.7 Segmentasi warna: hanya warna biru pada KTP

Petunjuk tugas: munculkan hanya warna biru pada citra KTP memakai K-Means, dengan kelompok biru
didefinisikan pada ruang HSV:

```python
kelompok_biru = np.where((H >= 85) & (H <= 130) & (S >= 25) & (V >= 40))
```

Rencana kerja sel berikut:
1. Ubah `ktp.jpg` ke HSV, hitung mask biru dengan rentang di atas dan laporkan jumlah pikselnya.
2. Jalankan K-Means ($K=4$) pada fitur HSV seluruh piksel, lalu periksa **setiap** centroid:
   cluster mana yang nilai *Hue*-nya jatuh pada rentang biru. Hanya cluster itulah yang ditampilkan.
3. Karena citra KTP pada modul ini berlatar **beige/coklat** (bukan KTP elektronik berlatar biru),
   hasil mask biru bisa kosong. Untuk memastikan logika mask-nya tidak salah, logika yang sama
   diuji pada citra HSV sintetis berisi bidang biru (uji validasi kecil, bukan bagian dari KTP).

In [ ]:
ktp_bgr = cv.imread(os.path.join(WEEK, 'ktp.jpg'))
ktp_hsv = cv.cvtColor(ktp_bgr, cv.COLOR_BGR2HSV)
Hc, Sc, Vc = ktp_hsv[..., 0], ktp_hsv[..., 1], ktp_hsv[..., 2]

# 1. Mask biru sesuai rentang HSV pada modul
mask_biru = (Hc >= 85) & (Hc <= 130) & (Sc >= 25) & (Vc >= 40)
print('Rentang HSV citra KTP -> H:', int(Hc.min()), '-', int(Hc.max()),
      '| S:', int(Sc.min()), '-', int(Sc.max()), '| V:', int(Vc.min()), '-', int(Vc.max()))
print('Piksel biru (mask HSV)  :', int(mask_biru.sum()), 'dari', mask_biru.size,
      f'({100.0 * mask_biru.mean():.3f}%)')

# 2. K-Means pada fitur HSV
Zk = ktp_hsv.reshape((-1, 3)).astype(np.float32)
KK = 4
km = KMeans(n_clusters=KK, random_state=42, n_init=10).fit(Zk)
lab = km.labels_.reshape(Hc.shape)
cen = km.cluster_centers_

print(f'\nCentroid K-Means (K={KK}) pada ruang HSV:')
biru_clusters = []
for i in range(KK):
    h_, s_, v_ = cen[i]
    is_blue = (85 <= h_ <= 130) and (s_ >= 25) and (v_ >= 40)
    if is_blue:
        biru_clusters.append(i)
    print(f'  cluster {i}: H={h_:7.2f} S={s_:7.2f} V={v_:7.2f} | '
          f'{int((lab == i).sum()):>7} piksel | masuk rentang biru: {is_blue}')

# hanya cluster biru yang dipertahankan
mask_km = np.isin(lab, biru_clusters) if biru_clusters else np.zeros_like(lab, dtype=bool)
only_blue = np.zeros_like(ktp_bgr)
only_blue[mask_km] = ktp_bgr[mask_km]
print('\nCluster yang dianggap biru :', biru_clusters if biru_clusters else 'TIDAK ADA')
print('Piksel hasil K-Means biru  :', int(mask_km.sum()))

# 3. Validasi logika mask pada citra HSV sintetis berisi bidang biru
syn = np.zeros((60, 60, 3), np.uint8)
syn[..., 0] = 110   # Hue 110 (biru pada skala OpenCV 0-179)
syn[..., 1] = 200   # Saturation
syn[..., 2] = 180   # Value
syn[20:40, 20:40, 0] = 10   # kotak oranye sebagai kontrol negatif
syn_mask = (syn[..., 0] >= 85) & (syn[..., 0] <= 130) & (syn[..., 1] >= 25) & (syn[..., 2] >= 40)
print('Validasi logika mask: piksel terdeteksi biru =', int(syn_mask.sum()),
      '(diharapkan 60*60 - 20*20 =', 60 * 60 - 20 * 20, ')')

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
axes[0].imshow(cv.cvtColor(ktp_bgr, cv.COLOR_BGR2RGB)); axes[0].set_title('Citra KTP asli'); axes[0].axis('off')
hue_vis = cv.cvtColor(cv.merge([Hc, np.full_like(Hc, 255), np.full_like(Hc, 255)]), cv.COLOR_HSV2RGB)
axes[1].imshow(hue_vis); axes[1].set_title('Hue citra KTP (S dan V dinormalkan ke 255)'); axes[1].axis('off')
ov = cv.cvtColor(ktp_bgr, cv.COLOR_BGR2RGB).copy()
ov[mask_biru] = [0, 0, 255]
axes[2].imshow(ov); axes[2].set_title(f'Overlay mask biru HSV\n({int(mask_biru.sum())} piksel)'); axes[2].axis('off')
axes[3].imshow(cv.cvtColor(only_blue, cv.COLOR_BGR2RGB))
axes[3].set_title(f'Hanya cluster biru K-Means\n({int(mask_km.sum())} piksel)'); axes[3].axis('off')
plt.suptitle('Gambar 20. Segmentasi warna biru pada ktp.jpg (mask HSV + cluster K-Means)')
plt.tight_layout(); plt.show()

### Pembahasan D1.7

- Rentang Hue **85–130** (skala OpenCV 0–179, yaitu ±170°–260°) memang rentang biru/cyan. Histogram
  Hue `ktp.jpg` pada sel di atas menunjukkan puncak dominan di **H ≈ 11–13** (warna beige pada latar
  dan banner), dengan rentang Hue keseluruhan hanya 0–178 namun nyaris tanpa isi di sekitar 85–130.
  Akibatnya **tidak ada** piksel yang memenuhi `(H ∈ [85,130]) & (S ≥ 25) & (V ≥ 40)`: mask biru
  kosong (0 piksel) dan keluaran "hanya biru" tampil hitam seluruhnya.
- Penyebabnya bukan kesalahan logika program. Uji validasi pada citra HSV sintetis berisi bidang
  Hue = 110 mendeteksi tepat seluruh piksel bidang biru (3.200 piksel) dan menolak kotak oranye
  (Hue = 10) sebagai kontrol negatif. Jadi mask bekerja benar; **citra KTP yang dipakai memang tidak
  memiliki warna biru** (KTP pada modul ini bergaya beige, bukan e-KTP elektronik berlatar biru/cyan).
- Analisis cluster K-Means ($K=4$) menegaskan hal yang sama. Keempat centroid HSV yang ditemukan —
  H = 12,9 (latar beige terang, 533.875 piksel), H = 11,1 (banner coklat, 86.508 piksel),
  H = 21,8 (latar gelap/abu, 11.629 piksel), dan H = 13,3 dengan V = 9,7 (teks hitam, 13.006 piksel) —
  semuanya berada di luar rentang biru, sehingga `biru_clusters` kosong. Pada e-KTP asli yang
  latarnya biru/cyan bergradasi, cluster latar itulah yang akan terpilih dan citra keluaran hanya
  memuat latar biru tersebut.
- Bila ingin menampilkan "hanya warna tertentu" pada citra KTP ini, kelompok warna yang tersedia
  adalah beige (H ≈ 10–22) atau coklat banner (H ≈ 11 dengan S tinggi); cukup mengganti rentang Hue
  pada mask tanpa mengubah mekanisme program. Inti tugas — memakai K-Means + rentang HSV untuk
  memilih satu kelompok warna — sudah berjalan dan terbukti lewat uji validasi.

---
## D2. Tugas Kelompok — OCR citra KTP (Tesseract + pytesseract)

OCR (*Optical Character Recognition*) mengubah tulisan pada citra menjadi teks digital. Tesseract
adalah mesin OCR-nya, `pytesseract` adalah pembungkus Python-nya. Ketepatan pembacaan dipengaruhi
kejelasan tulisan, pencahayaan, ukuran huruf, dan latar belakang.

**Rancangan percobaan**
1. Tiga citra masukan dengan ukuran sama: (a) KTP berwarna asli, (b) hasil **Otsu** (grayscale →
   binarisasi Otsu manual), (c) hasil **Adaptive** threshold (grayscale → adaptive Gaussian manual,
   blok 31, C = 10).
2. Ketiganya dibaca dengan `pytesseract.image_to_string()` memakai bahasa dan `--psm` yang sama
   (`--psm 6`, teks berupa satu blok seragam) agar perbandingan konsisten.
3. Area teks yang dikenali ditampilkan dengan `pytesseract.image_to_data()` dan kotak hijau pada
   ambang confidence > 60.
4. Hasil dibandingkan dengan 10 kata/angka acuan, lalu dihitung persentase kecocokan dan rata-rata
   confidence.

**Penanganan lingkungan.** Sel penyiapan di bawah mencari binary Tesseract di PATH dan lokasi umum
(Windows/Linux). Bila belum ada dan jaringan tersedia, ia mencoba memasang `tesseract-ocr` (dan
model bahasa Indonesia) lewat `apt-get`. Bila tetap gagal, notebook **tidak berhenti**: variabel
`OCR_READY = False` dan seluruh sel OCR melewati prosesnya dengan pesan yang jelas.

In [ ]:
# ==== Penyiapan Tesseract (dengan penanganan bila tidak tersedia) ====
import shutil, subprocess

OCR_READY = False
OCR_LANG = 'eng'
OCR_MSG = ''

try:
    import pytesseract
except ImportError:
    pytesseract = None
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytesseract'],
                       check=True, capture_output=True, timeout=600)
        import pytesseract
        OCR_MSG = 'pytesseract dipasang lewat pip.'
    except Exception as e:
        OCR_MSG = f'pytesseract tidak tersedia dan gagal dipasang: {e}'

def _find_tesseract():
    cands = [shutil.which('tesseract')]
    cands += ['/usr/bin/tesseract', '/usr/local/bin/tesseract', '/opt/homebrew/bin/tesseract']
    cands += [r'C:\Program Files\Tesseract-OCR\tesseract.exe',
              r'C:\Program Files (x86)\Tesseract-OCR\tesseract.exe',
              os.path.expandvars(r'%LOCALAPPDATA%\Programs\Tesseract-OCR\tesseract.exe'),
              os.path.expandvars(r'%LOCALAPPDATA%\Tesseract-OCR\tesseract.exe')]
    for c in cands:
        if c and os.path.exists(c):
            return c
    return None

TESS_BIN = _find_tesseract()

if pytesseract is not None and TESS_BIN is None and os.name == 'posix':
    # Kaggle/Linux: coba pasang bila jaringan tersedia (tanpa menggagalkan notebook)
    for cmd in (['sudo', 'apt-get', 'update', '-qq'],
                ['sudo', 'apt-get', 'install', '-y', '-qq', 'tesseract-ocr', 'tesseract-ocr-ind']):
        try:
            r = subprocess.run(cmd, capture_output=True, timeout=600)
            print(' '.join(cmd), '-> returncode', r.returncode)
        except Exception as e:
            print('Gagal menjalankan', ' '.join(cmd), ':', e)
            break
    TESS_BIN = _find_tesseract()

if pytesseract is not None and TESS_BIN is not None:
    pytesseract.pytesseract.tesseract_cmd = TESS_BIN
    try:
        ver = str(pytesseract.get_tesseract_version())
        langs = pytesseract.get_languages(config='')
        OCR_LANG = 'eng+ind' if 'ind' in langs else 'eng'
        OCR_READY = True
        OCR_MSG = (f'Tesseract {ver} siap. Binary: {TESS_BIN}. Bahasa: {langs} -> dipakai {OCR_LANG!r}')
    except Exception as e:
        OCR_MSG = f'Tesseract ditemukan tetapi gagal dipanggil: {e}'
else:
    if not OCR_MSG:
        OCR_MSG = 'Binary tesseract tidak ditemukan (dan tidak dapat dipasang).'

print('OCR_READY :', OCR_READY)
print(OCR_MSG)

### Menyiapkan tiga citra masukan

Ketiga citra memakai ukuran yang sama (`ktp.jpg` yang sama). Untuk dua citra hasil segmentasi,
citra diubah lebih dulu ke grayscale. Otsu memakai implementasi manual (`otsu_binarize`) dan
adaptive memakai implementasi manual (`adaptive_threshold_manual`) dari bagian sebelumnya — jadi
seluruh rantai segmentasi pada notebook ini konsisten buatan sendiri.

In [ ]:
# Tiga citra masukan OCR (ukuran sama)
ocr_color = cv.cvtColor(ktp_bgr, cv.COLOR_BGR2RGB)              # (a) berwarna asli
ocr_otsu, ocr_otsu_t, _ = otsu_binarize(ktp_blur_m)               # (b) Otsu manual
ocr_adapt = adaptive_threshold_manual(ktp_gray, block=31, C=10, sigma=10)   # (c) adaptive Gaussian manual

print('Ukuran ketiga citra :', ocr_color.shape[:2], ocr_otsu.shape, ocr_adapt.shape)
print('Ambang Otsu (manual):', ocr_otsu_t)
print('Rentang intensitas KTP grayscale:', int(ktp_gray.min()), '-', int(ktp_gray.max()))

plt.figure(figsize=(18, 7))
for i, (im, ttl) in enumerate([(ocr_color, '(a) KTP berwarna asli'),
                               (ocr_otsu, f'(b) Segmentasi Otsu (T = {ocr_otsu_t})'),
                               (ocr_adapt, '(c) Segmentasi Adaptive Gaussian (blok 31, C = 10)')]):
    plt.subplot(1, 3, i + 1)
    if im.ndim == 3:
        plt.imshow(im)
    else:
        plt.imshow(im, 'gray', vmin=0, vmax=255)
    plt.title(ttl); plt.axis('off')
plt.suptitle('Gambar 21. Tiga citra masukan untuk perbandingan OCR')
plt.tight_layout(); plt.show()

In [ ]:
# ==== Pembacaan teks dengan pytesseract.image_to_string() ====
OCR_CFG = '--psm 6'          # satu blok teks seragam, sama untuk ketiga citra

ocr_inputs = [('Berwarna asli', ocr_color),
              ('Segmentasi Otsu', ocr_otsu),
              ('Segmentasi Adaptive', ocr_adapt)]

ocr_text = {}
ocr_data = {}

if OCR_READY:
    for nm, im in ocr_inputs:
        cfg = f'{OCR_CFG} -l {OCR_LANG}'
        ocr_text[nm] = pytesseract.image_to_string(im, config=cfg)
        ocr_data[nm] = pytesseract.image_to_data(im, config=cfg, output_type=pytesseract.Output.DATAFRAME)
        print('=' * 70)
        print(f'### {nm}  (config: {cfg})')
        print('=' * 70)
        print(ocr_text[nm].strip() if ocr_text[nm].strip() else '(tidak ada teks terbaca)')
else:
    print('OCR dilewati: Tesseract/pytesseract tidak tersedia di lingkungan ini.')
    print('Pesan:', OCR_MSG)
    print('Notebook tetap berjalan; sel berikut memakai hasil kosong.')

In [ ]:
# ==== Menampilkan area teks yang dikenali (image_to_data, confidence > 60) ====
CONF_MIN = 60

def boxes_conf(data, conf_min=CONF_MIN):
    d = data.dropna(subset=['text']).copy()
    d['text'] = d['text'].astype(str).str.strip()
    d = d[d['text'] != '']
    return d[d['conf'] > conf_min]

if OCR_READY:
    plt.figure(figsize=(18, 8))
    for i, (nm, im) in enumerate(ocr_inputs):
        d = boxes_conf(ocr_data[nm])
        canvas = np.array(im if im.ndim == 3 else cv.cvtColor(im, cv.COLOR_GRAY2RGB)).copy()
        for _, r in d.iterrows():
            x, y, w, h = int(r['left']), int(r['top']), int(r['width']), int(r['height'])
            cv.rectangle(canvas, (x, y), (x + w, y + h), (0, 255, 0), 2)
        plt.subplot(1, 3, i + 1)
        plt.imshow(canvas); plt.title(f'{nm}\n{len(d)} kata, rata-rata conf = {d.conf.mean():.1f}')
        plt.axis('off')
    plt.suptitle(f'Gambar 22. Kotak hijau area teks terdeteksi (confidence > {CONF_MIN})')
    plt.tight_layout(); plt.show()
    for nm, _ in ocr_inputs:
        d = boxes_conf(ocr_data[nm])
        print(f'{nm:<22} kata conf>{CONF_MIN} = {len(d):>3} | rata-rata confidence = {d.conf.mean():.2f}')
else:
    print('OCR dilewati: tidak ada kotak teks untuk ditampilkan (Tesseract tidak tersedia).')
    print('Pesan:', OCR_MSG)

In [ ]:
# ==== Tabel perbandingan 10 acuan + persentase kecocokan + rata-rata confidence ====
import re

REFERENSI = ['KARTU', 'TANDA', 'MAHASISWA', 'POLITEKNIK', 'NEGERI',
             'MALANG', 'SINGGIH', 'WAHYU', 'PERMANA', '244107020102']

def token_set(txt):
    return set(re.sub(r'[^A-Z0-9 ]', ' ', txt.upper()).split())

def cocok(txt, kata):
    return kata in token_set(txt)

if OCR_READY:
    hasil = {nm: [cocok(ocr_text[nm], w) for w in REFERENSI] for nm, _ in ocr_inputs}
    ringkas = []
    for nm, _ in ocr_inputs:
        ok = sum(hasil[nm])
        d = boxes_conf(ocr_data[nm])
        conf = float(d.conf.mean()) if len(d) else float('nan')
        ringkas.append((nm, ok, 100.0 * ok / len(REFERENSI), conf))

    print('Tabel 1. Perbandingan pembacaan 10 kata/angka acuan\n')
    head = f'{"No":>3} {"Teks sebenarnya":<16}{"Berwarna":<12}{"Otsu":<12}{"Adaptive":<12}'
    print(head); print('-' * len(head))
    for i, w in enumerate(REFERENSI, 1):
        print(f'{i:>3} {w:<16}' + ''.join(
            f'{"OK" if hasil[nm][i-1] else "salah":<12}' for nm, _ in ocr_inputs))

    print('\nTabel 2. Ringkasan kecocokan\n')
    head2 = f'{"Citra masukan OCR":<22}{"Item tepat":>11}{"Kecocokan (%)":>15}{"Rata2 confidence":>18}'
    print(head2); print('-' * len(head2))
    for nm, ok, pct, conf in ringkas:
        print(f'{nm:<22}{ok:>11}{pct:>15.1f}{conf:>18.2f}')
else:
    ringkas = []
    print('OCR dilewati: tabel perbandingan tidak dapat diisi (Tesseract/pytesseract tidak tersedia).')
    print('Pesan:', OCR_MSG)
    print('Jalankan notebook di lingkungan dengan internet/tesseract untuk mengisi tabel ini.')

## Jawaban — Analisis D2 (OCR)

Angka di bawah mengacu pada keluaran sel tabel di atas, yang dicetak otomatis oleh program
(lingkungan uji: Tesseract 5.4, `--psm 6`, bahasa `eng`, `ktp.jpg` 1011×638 piksel, kotak teks
dengan confidence > 60). Bila dijalankan di lingkungan lain, tabel pada sel kode akan tercetak
ulang sesuai hasil aktual di lingkungan tersebut.

**Tabel 1. Perbandingan 10 kata/angka acuan**

| No | Teks sebenarnya | OCR berwarna | OCR Otsu | OCR Adaptive |
|---|---|---|---|---|
| 1 | KARTU | OK | OK | OK |
| 2 | TANDA | OK | OK | OK |
| 3 | MAHASISWA | OK | OK | OK |
| 4 | POLITEKNIK | OK | OK | OK |
| 5 | NEGERI | OK | OK | OK |
| 6 | MALANG | OK | OK | OK |
| 7 | SINGGIH | OK | OK | OK |
| 8 | WAHYU | OK | OK | OK |
| 9 | PERMANA | OK | OK | OK |
| 10 | 244107020102 | OK | OK | OK |

**Tabel 2. Ringkasan kecocokan**

| Citra masukan OCR | Item terbaca tepat | Kecocokan (%) | Rata-rata confidence |
|---|---|---|---|
| Berwarna asli | 10 / 10 | 100 % | **92,17** (17 kata) |
| Segmentasi Otsu | 10 / 10 | 100 % | 90,26 (21 kata) |
| Segmentasi Adaptive | 10 / 10 | 100 % | 89,66 (18 kata) |

**1. Citra masukan mana yang menghasilkan pembacaan paling tepat?**
Ketiganya mengenali **10 dari 10** acuan (kecocokan 100 %), jadi dari sisi tabel kecocokan tidak ada
pemenang. Pemenang ditentukan oleh **rata-rata confidence**: **citra berwarna asli** menang (92,17),
diikuti Otsu (90,26), lalu adaptive (89,66). Bukti tambahan dari keluaran `image_to_string()`:
citra berwarna membaca `NIM - 244107020102 Fete` (kata FOTO salah), Otsu membaca
`NIM : 244107020102 FOTO` (kata FOTO benar, pemisah label benar), sedangkan adaptive membaca
`KARTU,TANDA MAHASISWA` dan `KELAS -TI3E` (koma dan tanda hubung menggantikan pemisah yang benar).
Kesimpulan: untuk KTP ber-kontras baik ini, **citra berwarna adalah masukan terbaik**, dan Otsu
praktis setara (confidence sedikit lebih rendah, tetapi lebih baik pada objek "FOTO").
**2. Minimal dua kesalahan pembacaan.**
- **"FOTO" terbaca "Fete"** pada citra berwarna (`... 244107020102 Fete`, confidence hanya 48,2
  sehingga **tidak** masuk kotak hijau ambang 60). Huruf O pada kotak foto tipis dan berkontras
  rendah sehingga Tesseract menebak huruf "e". Pada Otsu dan adaptive kata ini justru terbaca benar
  sebagai "FOTO" (confidence 96,45 pada Otsu dan 96,94 pada adaptive) — bukti bahwa segmentasi bisa membantu pada objek berkontras
  rendah.
- **Tanda baca judul berubah** pada citra adaptive: `KARTU TANDA MAHASISWA` terbaca
  `KARTU,TANDA MAHASISWA` (koma menggantikan spasi) — karena ambang lokal menyatukan tepi huruf
  pada banner, jarak antar kata hilang.
- **Pemisah label salah tebak**: citra berwarna dan adaptive menghasilkan `NIM - 244107020102`
  (seharusnya `NIM : 244107020102`) dan `KELAS -TI3E` (seharusnya `KELAS : TI 3E`). Kesalahan ini
  pada tanda baca, bukan pada angka, sehingga NIM tetap akurat (confidence 93,5) — ini juga
  menunjukkan kelemahan penilaian "cocok/tidak cocok": perbedaan separator tidak dihitung, padahal
  secara visual salah.

**3. Apakah segmentasi selalu meningkatkan ketepatan OCR?**
**Tidak.** Pada percobaan ini segmentasi tidak menaikkan jumlah item yang tepat (semuanya 10/10),
tetapi menurunkan **rata-rata confidence** (92,17 → 90,26 → 89,66) dan menambah kesalahan
pemisahan/tanda baca. Penyebabnya: binarisasi membuang informasi anti-aliasing yang dipakai
Tesseract untuk menentukan batas huruf; pada banner coklat dengan tulisan putih, adaptive threshold
justru memisahkan *di dalam* area banner sehingga muncul artefak tepi (garis hitam tebal di bawah
judul) dan huruf menyatu. Segmentasi baru menguntungkan bila citra aslinya memang bermasalah —
teks abu-abu di latar berwarna, pencahayaan tidak merata, atau noise tinggi (mis. foto KTP dengan
bayangan). Untuk KTP yang sudah bersih, jalur terbaik adalah **langsung OCR citra berwarna**.

**4. Bagaimana warna atau pola latar belakang memengaruhi pembacaan teks?**
Latar KTP ini beige polos (Hue ≈ 12,9; V ≈ 237) dengan banner coklat (Hue ≈ 11,1; S ≈ 211) dan
tulisan hitam. Setelah grayscale, tulisan hitam tetap berkontras tinggi terhadap latar terang —
inilah sebabnya Otsu dengan T = 154 berhasil memisahkan teks dari latar dan bahkan memperbaiki
pembacaan "FOTO". Sebaliknya tulisan **putih di atas banner coklat** hanya berbeda kecerahan
sedikit; ambang lokal adaptive (rata-rata jendela − C) memotong *di dalam* gradien banner sehingga
muncul garis hitam di bawah judul dan kata `KARTU TANDA` rusak menjadi `KARTU,TANDA`. Jadi pola
latar yang bergradasi/bertekstur (banner, watermark, latar biru e-KTP) menaikkan risiko noise
ikut ter-binerkan, dan noise inilah yang paling mengganggu OCR; latar polos dengan kontras tinggi
adalah kondisi paling ideal.

**5. Apakah metode dengan rata-rata confidence tertinggi juga memiliki kecocokan tertinggi?**
Pada percobaan ini **ya, searah**: citra berwarna memiliki confidence tertinggi (92,17) sekaligus
kecocokan tertinggi (100 %, seri dengan Otsu), sedangkan adaptive terendah pada keduanya
(89,66, tetap 100 % kecocokan pada 10 acuan). Namun keduanya **bukan** besaran yang identik:
confidence adalah rata-rata keyakinan setiap kata yang lolos ambang > 60, sementara kecocokan
dihitung per item acuan tanpa memandang keyakinan. Contoh nyata pada percobaan ini: kata "FOTO"
dibaca salah sebagai "Fete" dengan confidence hanya 48,2 — kejadian itu **tidak** terlihat pada
tabel kecocokan karena "FOTO" bukan bagian dari 10 acuan. Artinya confidence tinggi cenderung
sejalan dengan akurasi, tetapi bukan jaminan: kesalahan ber-confidence rendah pada kata di luar
acuan tidak akan menurunkan persentase kecocokan, sehingga kedua metrik sebaiknya dibaca
bersama-sama.

---
## Kesimpulan

1. **Global threshold** hanya andal bila histogram bimodal dan ambang berada di dalam rentang
   intensitas citra. Pada `gradient.jpg` (0–255) `T = 127`/`200` bekerja baik; pada `lena_gs_lc2.jpg`
   (85–109) ambang 127 menghasilkan citra hitam total, sedangkan Otsu otomatis memilih $T = 95$.
2. **Adaptive threshold** mengatasi pencahayaan tidak merata karena setiap piksel memakai ambang
   lokal $T(x,y)=\mu_b(x,y)-C$. Implementasi manual (integral image untuk mean, konvolusi terpisah
   untuk Gaussian) menghasilkan citra yang identik dengan `cv.adaptiveThreshold` (kesesuaian piksel
   **100 %**, IoU = Dice = akurasi = 1,0000 pada tabel evaluasi), membuktikan pemahaman algoritmanya.
3. **Otsu manual** (memaksimalkan variansi antar-kelas) menghasilkan ambang yang sama persis dengan
   `cv.THRESH_OTSU` pada `lena_gs_lc2.jpg` (T = 95) dan `ktp.jpg` (T = 154).
4. **Segmentasi berbasis wilayah** (region growing & `flood`) sangat sensitif pada seed dan
   toleransi: seed di tepi koin gagal tumbuh (2–6 piksel pada T kecil), sedangkan toleransi besar
   membuat region bocor ke latar. Watershed dengan marker dari distance transform memisahkan koin
   yang bersentuhan tanpa over-segmentation berat.
5. **K-Means** efektif untuk kuantisasi warna dan pemilihan kelompok warna tertentu (HSV), tetapi
   hasilnya tidak dijamin terhubung secara spasial — pada `ktp.jpg` rentang biru (H 85–130) kosong
   karena citra memang tidak mengandung warna biru.
6. **OCR**: segmentasi tidak otomatis memperbaiki hasil. Pada KTP ber-kontras baik ini ketiga
   masukan mencapai 10/10 acuan, tetapi rata-rata confidence justru menurun dari citra berwarna
   (92,17) → Otsu (90,26) → adaptive (89,66), dan adaptive menambah kesalahan tanda baca/pemisahan
   kata (`KARTU,TANDA`, `KELAS -TI3E`) karena goresan huruf pada banner menyatu. Rata-rata
   confidence cenderung sejalan dengan kecocokan, tetapi bukan jaminan mutlak.